In [1]:
import os

In [2]:
%pwd

'/home/vk/Desktop/Python_Code/Pytorch/NLP/Text_Summarizer_Project/research'

In [3]:
os.chdir("../")
%pwd

'/home/vk/Desktop/Python_Code/Pytorch/NLP/Text_Summarizer_Project'

In [4]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    data_path: Path 
    model_ckpt: Path
    num_train_epochs: int
    warmup_steps: int
    per_device_train_batch_size: int
    weight_decay: float
    logging_steps: int
    evaluation_strategy: str
    eval_steps: int
    save_steps: float
    gradient_accumulation_steps: int 
    

In [5]:
from textSummarizer.constants import *
from textSummarizer.utils.common import read_yaml, create_directories

In [6]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):
        
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])
    
    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer
        params = self.params.TrainingArguments

        create_directories([config.root_dir])

        model_trainer_config = ModelTrainerConfig(
            root_dir=config.root_dir,
            data_path = config.data_path,
            model_ckpt = config.model_ckpt,
            num_train_epochs = params.num_train_epoch,
            warmup_steps = params.warmup_steps,
            per_device_train_batch_size = params.per_device_train_batch_size,
            weight_decay = params.weight_decay,
            logging_steps= params.logging_steps,
            evaluation_strategy = params.evaluation_strategy, 
            eval_steps = params.evaluation_strategy, 
            save_steps = params.save_steps,
            gradient_accumulation_steps = params.gradient_accumulation_steps
            )
        
        return  model_trainer_config

In [ ]:
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorForSeq2Seq
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from datasets import load_dataset, load_from_disk

import torch

[2026-06-02 13:56:44,110: INFO: utils: Note: NumExpr detected 24 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 16.]
[2026-06-02 13:56:44,111: INFO: utils: NumExpr defaulting to 16 threads.]


In [8]:
import os
import torch
import mlflow
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config


    
    def train(self):
        os.environ["WANDB_DISABLED"] = "true"

        device = "cuda" if torch.cuda.is_available() else "cpu"

        tokenizer = AutoTokenizer.from_pretrained(self.config.model_ckpt)

        model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(
            self.config.model_ckpt,
            torch_dtype=torch.float16
            ).to(device)

    # Memory optimization
        model_pegasus.gradient_checkpointing_enable()

        seq2seq_data_collator = DataCollatorForSeq2Seq(
            tokenizer,
            model=model_pegasus
            )

        # Load dataset
        dataset_samsum_pt = load_from_disk(self.config.data_path)

        trainer_args = TrainingArguments(
            output_dir=self.config.root_dir,

            num_train_epochs=1,

            warmup_steps=500,

            per_device_train_batch_size=1,
            per_device_eval_batch_size=1,

            gradient_accumulation_steps=16,

            weight_decay=0.01,

            logging_steps=10,

            evaluation_strategy="steps",
            eval_steps=500,

            save_steps=1000000,
            bf16=True,
            fp16=False,

            dataloader_pin_memory=False,

            report_to="none",

            run_name="pegasus_summarizer_v1"
            )

        trainer = Trainer(
            model=model_pegasus,
            args=trainer_args,
            tokenizer=tokenizer,
            data_collator=seq2seq_data_collator,
            train_dataset=dataset_samsum_pt["train"],
            eval_dataset=dataset_samsum_pt["validation"]
        )

        mlflow.set_tracking_uri("sqlite:///mlflow.db")

        torch.cuda.empty_cache()

        trainer.train()

    # Save model
        model_pegasus.save_pretrained(
        os.path.join(self.config.root_dir, "pegasus-samsum-model")
        )

        tokenizer.save_pretrained(
            os.path.join(self.config.root_dir, "tokenizer")
            )

In [9]:
try:
    config = ConfigurationManager()
    model_trainer_config = config.get_model_trainer_config()
    model_trainer_config = ModelTrainer(config=model_trainer_config)
    model_trainer_config.train()
except Exception as e:
    raise e

[2026-06-02 13:56:45,777: INFO: common: yaml file: config/config.yaml loaded successfully ]
[2026-06-02 13:56:45,778: INFO: common: yaml file: params.yaml loaded successfully ]
[2026-06-02 13:56:45,779: INFO: common: created directory at: artifacts]
[2026-06-02 13:56:45,779: INFO: common: created directory at: artifacts/model_trainer]


Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-cnn_dailymail and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/home/vk/Desktop/Python_Code/Pytorch/pytlib/lib/python3.11/site-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


  0%|          | 0/920 [00:00<?, ?it/s]

{'loss': 225081950208.0, 'grad_norm': nan, 'learning_rate': 1.0000000000000002e-06, 'epoch': 0.01}
{'loss': 0.0, 'grad_norm': nan, 'learning_rate': 2.0000000000000003e-06, 'epoch': 0.02}
{'loss': 0.0, 'grad_norm': nan, 'learning_rate': 3e-06, 'epoch': 0.03}
{'loss': 0.0, 'grad_norm': nan, 'learning_rate': 4.000000000000001e-06, 'epoch': 0.04}
{'loss': 0.0, 'grad_norm': nan, 'learning_rate': 5e-06, 'epoch': 0.05}
{'loss': 0.0, 'grad_norm': nan, 'learning_rate': 6e-06, 'epoch': 0.07}
{'loss': 0.0, 'grad_norm': nan, 'learning_rate': 7.000000000000001e-06, 'epoch': 0.08}
{'loss': 0.0, 'grad_norm': nan, 'learning_rate': 8.000000000000001e-06, 'epoch': 0.09}
{'loss': 0.0, 'grad_norm': nan, 'learning_rate': 9e-06, 'epoch': 0.1}
{'loss': 0.0, 'grad_norm': nan, 'learning_rate': 1e-05, 'epoch': 0.11}
{'loss': 0.0, 'grad_norm': nan, 'learning_rate': 1.1000000000000001e-05, 'epoch': 0.12}
{'loss': 0.0, 'grad_norm': nan, 'learning_rate': 1.2e-05, 'epoch': 0.13}
{'loss': 0.0, 'grad_norm': nan, 'lear

  0%|          | 0/818 [00:00<?, ?it/s]

{'eval_loss': nan, 'eval_runtime': 21.2933, 'eval_samples_per_second': 38.416, 'eval_steps_per_second': 38.416, 'epoch': 0.54}
{'loss': 0.0, 'grad_norm': nan, 'learning_rate': 4.880952380952381e-05, 'epoch': 0.55}
{'loss': 0.0, 'grad_norm': nan, 'learning_rate': 4.761904761904762e-05, 'epoch': 0.56}
{'loss': 0.0, 'grad_norm': nan, 'learning_rate': 4.642857142857143e-05, 'epoch': 0.58}
{'loss': 0.0, 'grad_norm': nan, 'learning_rate': 4.523809523809524e-05, 'epoch': 0.59}
{'loss': 0.0, 'grad_norm': nan, 'learning_rate': 4.404761904761905e-05, 'epoch': 0.6}
{'loss': 0.0, 'grad_norm': nan, 'learning_rate': 4.2857142857142856e-05, 'epoch': 0.61}
{'loss': 0.0, 'grad_norm': nan, 'learning_rate': 4.166666666666667e-05, 'epoch': 0.62}
{'loss': 0.0, 'grad_norm': nan, 'learning_rate': 4.047619047619048e-05, 'epoch': 0.63}
{'loss': 0.0, 'grad_norm': nan, 'learning_rate': 3.928571428571429e-05, 'epoch': 0.64}
{'loss': 0.0, 'grad_norm': nan, 'learning_rate': 3.809523809523809e-05, 'epoch': 0.65}
{'l

Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 128, 'min_length': 32, 'num_beams': 8, 'length_penalty': 0.8, 'forced_eos_token_id': 1}


{'loss': 0.0, 'grad_norm': nan, 'learning_rate': 0.0, 'epoch': 1.0}
{'train_runtime': 1902.6864, 'train_samples_per_second': 7.743, 'train_steps_per_second': 0.484, 'train_loss': 2446542937.0434785, 'epoch': 1.0}


In [13]:
import torch
torch.cuda.empty_cache()

In [14]:
!nvidia-smi

Tue Jun  2 14:52:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.03             Driver Version: 580.159.03     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4050 ...    Off |   00000000:01:00.0  On |                  N/A |
| N/A   49C    P0             11W /  140W |    1332MiB /   6141MiB |     72%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----